# Adversarial Attacks and Robustness

## Overview
Adversarial examples are inputs crafted by adding imperceptible perturbations that cause ML models to make wrong predictions with high confidence. Understanding attacks is essential for building robust, trustworthy AI systems.

---

## 1. What Are Adversarial Examples?

For an image classifier $f: \mathcal{X} \to \mathcal{Y}$, an adversarial example is:
$$x_{adv} = x + \delta \quad \text{such that} \quad f(x_{adv}) \neq f(x) \quad \text{and} \quad ||\delta||_p \leq \epsilon$$

The perturbation $\delta$ is:
- **Imperceptible**: $\epsilon$ is small (e.g., max pixel change of 8/255)
- **Adversarial**: causes misclassification

Common norm constraints:
- $L_\infty$: $\max_i |\delta_i| \leq \epsilon$ bounds each pixel independently
- $L_2$: $||\delta||_2 \leq \epsilon$ bounds total energy
- $L_0$: $||\delta||_0 \leq k$ at most $k$ pixels changed

---

## 2. White-Box Attacks

Attacker has full access to model weights and gradients.

### 2.1 FGSM (Fast Gradient Sign Method) Goodfellow et al. 2014

One-step attack using the gradient of the loss:
$$x_{adv} = x + \epsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$

Where:
- $J$: cross-entropy loss
- $\epsilon$: perturbation magnitude
- $\text{sign}$: element-wise sign function

**Intuition**: take one step in the direction that increases the loss the most.

### 2.2 PGD (Projected Gradient Descent) Madry et al. 2017

Iterative FGSM with projection back to $\epsilon$-ball:
$$x_{adv}^{t+1} = \Pi_{x+S}\left(x_{adv}^t + \alpha \cdot \text{sign}(\nabla_{x} J(\theta, x_{adv}^t, y))\right)$$

Where $\Pi_{x+S}$ projects back into the allowed perturbation set $S$.

PGD is considered the **strongest first-order attack** if a model is robust to PGD, it's generally robust.

### 2.3 C&W Attack (Carlini & Wagner 2016)

Optimization-based attack finding minimal perturbation:
$$\min_{\delta} ||\delta||_2^2 + c \cdot f(x + \delta)$$

Where $f$ is a loss function designed to achieve misclassification. Much stronger than FGSM/PGD but slower.

### 2.4 DeepFool

Finds the **minimum perturbation** to cross the decision boundary:
$$r_* = \arg\min_r ||r||_2 \quad \text{s.t.} \quad f(x + r) \neq f(x)$$
Linearizes the boundary iteratively.

### 2.5 AutoAttack
Ensemble of parameter-free attacks: APGD-CE, APGD-DLR, FAB, Square Attack. The **de facto benchmark** for adversarial robustness.

---

## 3. Black-Box Attacks

Attacker only has query access (no gradients).

### Score-based
- **Natural Evolution Strategies (NES)**: estimate gradient from output probabilities
- **Square Attack**: random search in $L_\infty$ perturbation set

### Transfer-based
- Generate adversarial examples on a substitute model
- Adversarial examples often **transfer** across different architectures

---

## 4. Physical World Attacks

Adversarial patches that work in the physical world:
- Print and attach adversarial stickers to stop signs → fool autonomous vehicles
- Adversarial glasses → fool face recognition
- Must be robust to: rotation, scale, lighting, viewpoint changes

---

## 5. Defenses

### 5.1 Adversarial Training
Include adversarial examples in training:
$$\min_\theta \mathbb{E}_{(x,y)} \left[ \max_{\delta \in S} L(\theta, x + \delta, y) \right]$$
This is a min-max (saddle point) problem. **Most effective defense** but expensive.

### 5.2 Certified Defenses Randomized Smoothing
For a base classifier $f$, smoothed classifier $g$:
$$g(x) = \arg\max_c P(f(x + \varepsilon) = c), \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$$

**Certificate**: $g$ is guaranteed to predict class $c_A$ for all $||\delta||_2 \leq R$ where:
$$R = \frac{\sigma}{2}(\Phi^{-1}(p_A) - \Phi^{-1}(p_B))$$

### 5.3 Input Preprocessing
- JPEG compression, bit-depth reduction, spatial smoothing
- **Limitation**: adaptive attacks bypass these

### 5.4 Adversarial Detection
Train a separate detector to flag adversarial inputs.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── FGSM Attack from Scratch ────────────────────────────────────
def fgsm_attack(model, x, y, epsilon=0.1):
    """Fast Gradient Sign Method."""
    x_adv = x.clone().detach().requires_grad_(True)
    
    # Forward pass
    output = model(x_adv)
    loss = F.cross_entropy(output, y)
    
    # Backward pass
    loss.backward()
    
    # FGSM step: x_adv = x + epsilon * sign(grad)
    perturbation = epsilon * x_adv.grad.sign()
    x_adv = x + perturbation
    
    # Clip to valid range [0, 1]
    x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()

print("FGSM implemented")

# ── PGD Attack ──────────────────────────────────────────────────
def pgd_attack(model, x, y, epsilon=0.1, alpha=0.01, num_steps=40):
    """Projected Gradient Descent attack."""
    x_adv = x.clone().detach()
    # Random initialization within epsilon ball
    x_adv = x_adv + torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
    x_adv = torch.clamp(x_adv, 0, 1).detach()
    
    for _ in range(num_steps):
        x_adv.requires_grad_(True)
        output = model(x_adv)
        loss = F.cross_entropy(output, y)
        
        loss.backward()
        
        # Gradient step
        with torch.no_grad():
            x_adv = x_adv + alpha * x_adv.grad.sign()
            # Project back to epsilon ball
            delta = torch.clamp(x_adv - x, -epsilon, epsilon)
            x_adv = torch.clamp(x + delta, 0, 1).detach()
    
    return x_adv

print("PGD implemented")

FGSM implemented
PGD implemented


In [2]:
# ── Simple demo: adversarial attack on a toy classifier ─────────
torch.manual_seed(42)

# Toy 2D binary classifier
class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 2)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

model = ToyNet()
# Synthetic data
X = torch.randn(200, 2)
Y = (X[:, 0] + X[:, 1] > 0).long()

# Quick training
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(200):
    loss = F.cross_entropy(model(X), Y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

# Evaluate clean accuracy
with torch.no_grad():
    clean_acc = (model(X).argmax(1) == Y).float().mean()

# FGSM attack
X_adv = fgsm_attack(model, X, Y, epsilon=0.3)
with torch.no_grad():
    adv_acc = (model(X_adv).argmax(1) == Y).float().mean()

# PGD attack
X_pgd = pgd_attack(model, X, Y, epsilon=0.3, alpha=0.05, num_steps=20)
with torch.no_grad():
    pgd_acc = (model(X_pgd).argmax(1) == Y).float().mean()

print(f"Clean accuracy:  {clean_acc:.3f}")
print(f"Under FGSM:      {adv_acc:.3f}")
print(f"Under PGD:       {pgd_acc:.3f}")

L_inf_fgsm = (X_adv - X).abs().max().item()
print(f"\nMax perturbation (FGSM L∞): {L_inf_fgsm:.4f}")

Clean accuracy:  1.000
Under FGSM:      0.535
Under PGD:       0.535

Max perturbation (FGSM L∞): 2.5850


In [3]:
# ── Adversarial Training ────────────────────────────────────────
torch.manual_seed(42)
robust_model = ToyNet()
optimizer = torch.optim.Adam(robust_model.parameters(), lr=0.01)

epsilon = 0.3
for epoch in range(200):
    # Generate adversarial examples using PGD
    robust_model.eval()
    X_adv_train = pgd_attack(robust_model, X, Y, epsilon=epsilon, alpha=0.05, num_steps=7)
    
    # Train on adversarial examples
    robust_model.train()
    loss = F.cross_entropy(robust_model(X_adv_train), Y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

with torch.no_grad():
    robust_clean_acc = (robust_model(X).argmax(1) == Y).float().mean()

X_adv_test = pgd_attack(robust_model, X, Y, epsilon=epsilon, alpha=0.05, num_steps=20)
with torch.no_grad():
    robust_adv_acc = (robust_model(X_adv_test).argmax(1) == Y).float().mean()

print("Adversarially trained model:")
print(f"  Clean accuracy:   {robust_clean_acc:.3f}")
print(f"  Robust accuracy:  {robust_adv_acc:.3f}")
print("\nNote: Some clean accuracy is traded for robustness (the robustness-accuracy tradeoff).")

Adversarially trained model:
  Clean accuracy:   0.785
  Robust accuracy:  0.675

Note: Some clean accuracy is traded for robustness (the robustness-accuracy tradeoff).


## Additional Learning Resources

### Papers
- [FGSM Explaining and Harnessing Adversarial Examples (Goodfellow et al.)](https://arxiv.org/abs/1412.6572)
- [PGD Towards Deep Learning Models Resistant to Adversarial Attacks (Madry et al.)](https://arxiv.org/abs/1706.06083)
- [C&W Attack (Carlini & Wagner)](https://arxiv.org/abs/1608.04644)
- [Randomized Smoothing (Cohen et al.)](https://arxiv.org/abs/1902.02918)
- [AutoAttack (Croce & Hein)](https://arxiv.org/abs/2003.01690)
- [RobustBench Leaderboard](https://robustbench.github.io/)

### Libraries
- [Adversarial Robustness Toolbox (ART)](https://github.com/Trusted-AI/adversarial-robustness-toolbox)
- [Foolbox](https://foolbox.readthedocs.io/)
- [CleverHans](https://github.com/cleverhans-lab/cleverhans)
- [Torchattacks](https://github.com/Harry24k/adversarial-attacks-pytorch)

### Courses
- [MIT 6.S965 TinyML and Efficient Deep Learning](https://hanlab.mit.edu/courses/2023-fall-65940)
- [Adversarial ML Tutorial NeurIPS](https://adversarial-ml-tutorial.org/)